In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("silver_schema", "lab8_dev_silver")
dbutils.widgets.text("gold_schema", "lab8_dev_gold")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

silver = f"{catalog}.{silver_schema}"
gold = f"{catalog}.{gold_schema}"

print(f"Silver: {silver}")
print(f"Gold: {gold}")

In [ ]:

# Gold Layer - adapted from Lab 6
# requires silver tables and dim_country_scd1

spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.dim_country
USING DELTA AS
SELECT
    country_code,
    AREA AS country_name,
    region,
    subregion
FROM {silver}.dim_country_scd1
WHERE AREA <> 'Russian Federation'
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.dim_crop
USING DELTA AS
SELECT
    ROW_NUMBER() OVER (ORDER BY ITEM) AS crop_key,
    ITEM AS crop_name
FROM (
    SELECT DISTINCT TRIM(ITEM) AS ITEM
    FROM {silver}.yield_and_climate
    WHERE ITEM IS NOT NULL
      AND TRIM(ITEM) <> ''
)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.dim_year
USING DELTA AS
SELECT DISTINCT
    CAST(YEAR AS INT) AS year_key,
    CAST(YEAR AS INT) AS year
FROM {silver}.yield_and_climate
WHERE YEAR IS NOT NULL
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.fact_agri_environment
USING DELTA AS
SELECT
    c.country_code AS country_key,
    cr.crop_key,
    y.year_key,
    a.YIELD AS crop_yield,
    a.AREA_HARVESTED AS area_harvested,
    a.PRODUCTION_QUANTITY AS production_quantity,
    e.freshwater_per_capita,
    e.freshwater_withdrawals,
    e.agricultural_land,
    e.forest_area
FROM {silver}.yield_and_climate a
INNER JOIN {gold}.dim_country c
    ON TRIM(a.AREA) = c.country_name
INNER JOIN {gold}.dim_crop cr
    ON TRIM(a.ITEM) = cr.crop_name
INNER JOIN {gold}.dim_year y
    ON CAST(a.YEAR AS INT) = y.year_key
LEFT JOIN {silver}.environment e
    ON TRIM(a.AREA) = TRIM(e.AREA)
   AND CAST(a.YEAR AS INT) = CAST(e.YEAR AS INT)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.agg_crop_year
USING DELTA AS
SELECT
    f.year_key AS year,
    c.crop_name,
    COUNT(DISTINCT f.country_key) AS country_count,
    SUM(f.production_quantity) AS total_production,
    SUM(f.area_harvested) AS total_area_harvested,
    CASE
        WHEN SUM(f.area_harvested) > 0
        THEN SUM(f.production_quantity) / SUM(f.area_harvested)
        ELSE NULL
    END AS production_per_harvested_area,
    AVG(f.crop_yield) AS avg_reported_yield
FROM {gold}.fact_agri_environment f
JOIN {gold}.dim_crop c
    ON f.crop_key = c.crop_key
GROUP BY f.year_key, c.crop_name
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold}.agg_country_year
USING DELTA AS
SELECT
    f.country_key,
    c.country_name,
    c.region,
    c.subregion,
    f.year_key AS year,
    COUNT(*) AS crop_count,
    SUM(f.production_quantity) AS total_production,
    SUM(f.area_harvested) AS total_area_harvested,
    CASE
        WHEN SUM(f.area_harvested) > 0
        THEN SUM(f.production_quantity) / SUM(f.area_harvested)
        ELSE NULL
    END AS production_per_harvested_area,
    MAX(f.freshwater_per_capita) AS freshwater_per_capita,
    MAX(f.freshwater_withdrawals) AS freshwater_withdrawals,
    MAX(f.agricultural_land) AS agricultural_land,
    MAX(f.forest_area) AS forest_area
FROM {gold}.fact_agri_environment f
JOIN {gold}.dim_country c
    ON f.country_key = c.country_code
GROUP BY
    f.country_key,
    c.country_name,
    c.region,
    c.subregion,
    f.year_key
""")

print("Gold tables created/refreshed successfully")
